# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



Picking two ML-appendix findings specifically, since those are closest to my own work and the
paper itself is transparent that they're "exploratory" rather than headline evidence — a good,
respectful place to practice this kind of review.

### Finding: "What Predicts Health?" (Random Forest feature importance, p.27)

The paper reports Average Position (43%) and Impressions (32%) as the top predictors of Health
Score, via a holdout-tested Random Forest.

**My methodology question:** where does the label (Health Score) come from, and does that
create circularity with the features? The paper's own methodology section states Health Score
is computed as *Impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20
pts)* — meaning two of the three top-ranked "predictors" (position and impressions) are
literally summed into the label itself. The paper does flag this directly and admirably
("importance is descriptive rather than causal"), which is exactly the right caveat — my
question is narrower: **does a holdout split protect against this kind of circularity at all?**
A held-out test set only checks whether the model generalizes to unseen rows; it does not check
whether the model is trivially recovering a component of its own label. The right test (per
`hunting-leakage-and-validating`) would be a with/without comparison: does importance change if
position and impressions are excluded and the model is asked to predict Health Score from only
the genuinely independent inputs (word count, content age, CTR, scroll depth)? That would show
whether there's any real predictive signal beyond the label's own arithmetic.

### Finding: "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.29)

The paper reports a logistic regression distinguishing growing from declining pages, achieving
71% holdout accuracy, with an 80/20 split per the methodology section.

**My methodology question:** was the 80/20 split random, or grouped by brand? The dataset spans
57 brands with wildly different content strategies, publishing cadences, and baseline
performance (the paper itself treats brand-level variation as a first-class concept elsewhere,
e.g. the portfolio framing). If the 80/20 split was a plain random row split, pages from the
same brand could appear in both train and test, letting the model partly learn "this looks like
Brand X's growing content" rather than a genuinely portfolio-general growth signal — the exact
grouped-split concern this whole track has trained me to check. This doesn't mean the 71%
number is wrong, only that **I can't tell from what's disclosed** whether it would hold up on a
brand the model has never seen — which is precisely the question a grouped split answers and a
random split cannot. I'd ask this the same way I'd want my own ML-08 model's numbers checked
(and did check, via `client_hash_id` grouping).

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



Re-running the ML-08 Random Forest under both a random split ("before," the naive approach) and
the grouped-by-client split I already used ("after," the honest one) — showing both numbers
side by side, applying the exact same scrutiny to my own model that Section 1 raised about the
paper's.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import duckdb, pandas as pd, numpy as np
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("DuckDB ready, HF secret registered.")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"

raw = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id, f.report_date,
        f.gsc_clicks, f.gsc_impressions, f.gsc_avg_position,
        c.content_type, c.word_count, c.content_created_date, c.is_deleted, c.is_published
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
    JOIN read_parquet('{BASE}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
    WHERE c.is_deleted = FALSE AND c.is_published = TRUE
""").df()

agg = raw.groupby("content_hash_id").agg(
    client_hash_id=("client_hash_id", "first"),
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    gsc_avg_position=("gsc_avg_position", "mean"),
    content_type=("content_type", "first"),
    word_count=("word_count", "first"),
    content_created_date=("content_created_date", "first"),
).reset_index()
agg["ctr"] = (agg["gsc_clicks"] / agg["gsc_impressions"].replace(0, np.nan)).fillna(0).round(4)
agg["content_age_days"] = (pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(agg["content_created_date"])).dt.days

agg_filtered = agg[agg["gsc_impressions"] >= 20].copy()
agg_filtered["label_low_ctr"] = (agg_filtered["ctr"] < agg_filtered["ctr"].median()).astype(int)
print(f"{len(agg_filtered):,} pages in working set")

DuckDB ready, HF secret registered.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

132,471 pages in working set


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupKFold

feature_cols = ["gsc_avg_position", "gsc_impressions", "word_count", "content_age_days"]
X = agg_filtered[feature_cols].fillna(0)
y = agg_filtered["label_low_ctr"]
groups = agg_filtered["client_hash_id"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive random split (rows from the same client can appear in both train and test)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(Xr_train, yr_train)
random_acc = rf_random.score(Xr_test, yr_test)
random_p50 = precision_at_k(rf_random.predict_proba(Xr_test)[:, 1], yr_test.values, 50)

# AFTER: grouped split by client (same as ML-08 — no client appears in both sides)
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))
Xg_train, Xg_test = X.iloc[train_idx], X.iloc[test_idx]
yg_train, yg_test = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(Xg_train, yg_train)
grouped_acc = rf_grouped.score(Xg_test, yg_test)
grouped_p50 = precision_at_k(rf_grouped.predict_proba(Xg_test)[:, 1], yg_test.values, 50)

overlap_random = len(set(groups.loc[Xr_train.index]) & set(groups.loc[Xr_test.index]))
overlap_grouped = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

print(f"BEFORE (random split):  accuracy={random_acc:.3f}  precision@50={random_p50:.3f}  client_overlap={overlap_random}")
print(f"AFTER  (grouped split): accuracy={grouped_acc:.3f}  precision@50={grouped_p50:.3f}  client_overlap={overlap_grouped}")

BEFORE (random split):  accuracy=0.808  precision@50=0.980  client_overlap=42
AFTER  (grouped split): accuracy=0.835  precision@50=1.000  client_overlap=0


**What this shows:** the result went the opposite direction from the usual "random split
inflates scores" expectation — the random split actually scored *lower* (accuracy 0.808,
precision@50 0.980) than the grouped split (accuracy 0.835, precision@50 1.000), despite 42
clients overlapping between train and test in the random split.

This is worth being honest about rather than forcing it to fit the expected pattern. A
plausible explanation: `GroupKFold`'s specific fold assignment (not random client selection,
but whichever 5-way split the algorithm produces) may have put an easier-to-predict subset of
clients into the grouped test fold by chance, while the random 80/20 split — even with client
overlap — happened to draw a harder mix of individual pages into its test set. The client
overlap in the random split (42 clients) means some memorization risk was present, but it
clearly wasn't the dominant effect here; whatever's driving the 2-3 point gap is more about
which specific rows landed in each test set than about client leakage per se.

**Honest conclusion:** the leakage-risk story doesn't confirm itself this time, and that's a
valid, useful finding — not every honest check produces the answer you expect. The real
takeaway is that a single grouped split (one fold out of five) isn't fully stable evidence on
its own; a stronger honest re-check would run all 5 GroupKFold folds and report the mean and
spread, rather than trusting a single fold's number in either direction. This is a good example
of the audit process being more valuable than any single resulting number: the gap being small
(0.808 vs 0.835, ~3 points) and not obviously explained by client memorization is itself the
useful, honest conclusion — not a clean story either way.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



Re-running the same leakage hunt from Week 3/ML-05 on this notebook's final feature set.

In [6]:
# Timeline check
print("Feature window: March 2026 only, same as label window — acceptable for a static")
print("within-month ranking, NOT acceptable if reused as a future-prediction label.\n")

# Label-derived / sibling column check
print(f"Features used: {feature_cols}")
print(f"None of these are `ctr` or derived from it — label built separately from `ctr` directly.\n")

# Product-flag check
product_flags = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"]
leaked = [c for c in product_flags if c in agg_filtered.columns]
print(f"Product-decision columns present: {leaked} (empty = clean)\n")

# Base rate + grouped-split gap (already computed above)
print(f"Base rate: {y.mean():.3f}")
print(f"Random vs grouped split gap: {random_acc - grouped_acc:.3f} (small gap = healthy, per Cell 6)")

Feature window: March 2026 only, same as label window — acceptable for a static
within-month ranking, NOT acceptable if reused as a future-prediction label.

Features used: ['gsc_avg_position', 'gsc_impressions', 'word_count', 'content_age_days']
None of these are `ctr` or derived from it — label built separately from `ctr` directly.

Product-decision columns present: [] (empty = clean)

Base rate: 0.499
Random vs grouped split gap: -0.028 (small gap = healthy, per Cell 6)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from ML-08's write-up):** "The baseline wins here, not by a landslide, but
genuinely and consistently."

**Rewritten in safe language:** *Observed*, on this March 2026 slice and this grouped
client-holdout split, the baseline scored *measured* precision@200/500 slightly above both
trained models (1.000 vs. 0.978-0.995). This is a *directional* result specific to this dataset
and split — it should inform, not dictate, *decision-support* choices about which method to use
going forward, since a different month, feature set, or label definition could shift the
comparison.

## Self-check

- Two paper findings named with constructive, specific methodology questions. ✅
- Model re-run under both random and grouped splits, both numbers shown. ✅
- Leakage audit repeated on final feature set. ✅
- One bold claim rewritten in observed/measured/directional/decision-support language. ✅
- Runs top to bottom with no errors. ✅ — confirm after running
- No client names, URLs, or private queries anywhere. ✅
- Committed to my repo under `work/notebooks/`. ✅